# Tutorial 04 -- Qubit Drive and Basic Population Dynamics

Drive a resonant qubit pulse in the matched rotating frame and inspect the time-domain populations of `|g>` and `|e>`.

**Prerequisites.** Tutorials 01 and 02 are recommended first.


## 1. Goal

We will apply a resonant square pulse, store the trajectory, and verify that a calibrated pulse produces the expected population transfer.


## 2. Physical Background

In the repository's square-pulse drive convention, a resonant two-level qubit evolves as `P_e(t) = sin^2(Omega t)`, so an ideal `pi` pulse occurs at `t_pi = pi / (2 Omega)`. This notebook checks that the pulse-level simulation follows that normalization explicitly.


## 3. Imports


In [ ]:
from __future__ import annotations

from functools import partial
from pathlib import Path
import sys

REPO_ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "pyproject.toml").exists() and (candidate / "cqed_sim").is_dir()
    ),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError("Could not resolve the repository root from the notebook working directory.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import qutip as qt

from cqed_sim import (
    AmplifierChain,
    BosonicModeSpec,
    DispersiveCouplingSpec,
    DispersiveReadoutTransmonStorageModel,
    DispersiveTransmonCavityModel,
    DisplacementGate,
    FrameSpec,
    NoiseSpec,
    Pulse,
    PurcellFilter,
    QubitMeasurementSpec,
    ReadoutChain,
    ReadoutResonator,
    RotationGate,
    SidebandDriveSpec,
    SequenceCompiler,
    SimulationConfig,
    StatePreparationSpec,
    TransmonModeSpec,
    UniversalCQEDModel,
    build_displacement_pulse,
    build_rotation_pulse,
    build_sideband_pulse,
    carrier_for_transition_frequency,
    coherent_state,
    compute_energy_spectrum,
    fock_state,
    manifold_transition_frequency,
    measure_qubit,
    prepare_simulation,
    prepare_state,
    pure_dephasing_time_from_t1_t2,
    qubit_state,
    run_rabi,
    run_ramsey,
    run_spectroscopy,
    run_t1,
    run_t2_echo,
    sideband_transition_frequency,
    simulate_batch,
    simulate_sequence,
)
from cqed_sim.plotting import plot_energy_levels
from cqed_sim.pulses import gaussian_envelope, square_envelope
from cqed_sim.sim import (
    cavity_wigner,
    conditioned_bloch_xyz,
    mode_moments,
    qubit_conditioned_mode_moments,
    readout_response_by_qubit_state,
    reduced_cavity_state,
    reduced_qubit_state,
    reduced_storage_state,
    storage_photon_number,
    subsystem_level_population,
    transmon_level_populations,
)
from tutorials.tutorial_support import (
    GHz,
    MHz,
    angular_to_ghz,
    angular_to_hz,
    angular_to_mhz,
    cross_kerr_conditional_phase,
    final_expectation,
    fit_echo_signal,
    fit_exponential_decay,
    fit_lorentzian_peak,
    fit_rabi_vs_amplitude,
    fit_rabi_vs_duration,
    fit_ramsey_signal,
    gaussian_quasistatic_echo_excited_population,
    gaussian_quasistatic_ramsey_excited_population,
    ns,
    ramsey_population,
    resonant_drive_excited_population,
    t1_relaxation_population,
    us,
)

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (7.0, 4.2)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False


## 4. Simulation Parameters


In [ ]:
omega_rabi = 2.0 * np.pi * 8.0e6
pulse_duration = np.pi / (2.0 * omega_rabi)
dt = pulse_duration / 200.0


## 5. Model Construction


In [ ]:
model = DispersiveTransmonCavityModel(
    omega_c=GHz(5.0),
    omega_q=GHz(6.1),
    alpha=0.0,
    chi=0.0,
    kerr=0.0,
    n_cav=1,
    n_tr=2,
)
frame = FrameSpec(omega_q_frame=model.omega_q)
initial_state = model.basis_state(0, 0)


## 6. Pulse / Sequence Construction


In [ ]:
pulse = Pulse(
    "q",
    t0=0.0,
    duration=pulse_duration,
    envelope=square_envelope,
    amp=omega_rabi,
    carrier=0.0,
    label="resonant_pi",
)
compiled = SequenceCompiler(dt=dt).compile([pulse], t_end=pulse_duration + dt)


## 7. Running the Simulation


In [ ]:
result = simulate_sequence(
    model,
    compiled,
    initial_state,
    {"q": "qubit"},
    config=SimulationConfig(frame=frame, store_states=True, max_step=dt),
)
trajectory_t = np.asarray(compiled.tlist, dtype=float)
trajectory_t_ns = trajectory_t * 1.0e9
p_e = np.asarray(result.expectations["P_e"], dtype=float)
p_g = np.asarray(result.expectations["P_g"], dtype=float)
theory_p_e = resonant_drive_excited_population(omega_rabi, trajectory_t)
theory_p_g = 1.0 - theory_p_e
print(f"Final excited-state population: {p_e[-1]:.4f}")
print(f"Maximum |simulation - theory| for P_e: {np.max(np.abs(p_e - theory_p_e)):.3e}")


## 8. Visualizing the Results


In [ ]:
fig, ax = plt.subplots()
ax.plot(trajectory_t_ns, p_g, label=r"simulation $P_g$")
ax.plot(trajectory_t_ns, p_e, label=r"simulation $P_e$")
ax.plot(trajectory_t_ns, theory_p_g, "--", color="tab:blue", alpha=0.8, label=r"theory $1 - \sin^2(\Omega t)$")
ax.plot(trajectory_t_ns, theory_p_e, "--", color="tab:orange", alpha=0.8, label=r"theory $\sin^2(\Omega t)$")
ax.set_xlabel("Time [ns]")
ax.set_ylabel("Population")
ax.set_title("Resonant qubit drive in the matched rotating frame")
ax.legend()
plt.show()


## 9. Physical Interpretation

Because the frame is matched and the drive is resonant, the simulated populations follow the expected `sin^2(Omega t)` exchange. This is the foundational normalization used again in the power-Rabi and time-Rabi tutorials later in the curriculum.


## 10. Exercises / Next Steps

- Halve the pulse duration and confirm that the final state is close to an `x90` rotation instead of a `pi` rotation.
- Add a small detuning through `carrier_for_transition_frequency(...)` and observe the loss of full inversion.
- Continue to Tutorials 09 and 10 for calibration-style Rabi sweeps.
